In [1]:
'''
!pip install pymupdf --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install sentence-transformers --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install faiss-cpu --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install google-generativeai --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install pandas --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install numpy --trusted-host pypi.org --trusted-host files.pythonhosted.org
!pip install tqdm --trusted-host pypi.org --trusted-host files.pythonhosted.org
'''

'\n!pip install pymupdf --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install sentence-transformers --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install faiss-cpu --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install google-generativeai --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install pandas --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install numpy --trusted-host pypi.org --trusted-host files.pythonhosted.org\n!pip install tqdm --trusted-host pypi.org --trusted-host files.pythonhosted.org\n'

In [2]:
# ==========================================
# Imports
# ==========================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

from tqdm import tqdm

In [3]:
# ==========================================
# Read All PDFs
# ==========================================

base_path = Path("../Courses")

pdf_files = list(base_path.rglob("*.pdf"))

print(f"Total PDFs : {len(pdf_files)}")

Total PDFs : 16


In [4]:
# ==========================================
# Build Dataset
# ==========================================

dataset = []

for pdf in pdf_files:

    dataset.append(
        {
            "course": pdf.parent.name,
            "file_name": pdf.name,
            "path": str(pdf)
        }
    )

df = pd.DataFrame(dataset)

df.head()

,course,file_name,path
0,AI Tools,Lec1 CV DIP Fundamentals.pdf,..\Courses\AI Tools\Lec1 CV DIP Fundamentals.pdf
1,AI Tools,Lec2 CV DIP Fundamentals.pdf,..\Courses\AI Tools\Lec2 CV DIP Fundamentals.pdf
2,AI Tools,Lec3 CV DIP Fundamentals PI II.pdf,..\Courses\AI Tools\Lec3 CV DIP Fundamentals P...
3,AI Tools,Lec4 Convolution Neural Networks.pdf,..\Courses\AI Tools\Lec4 Convolution Neural Ne...
4,AI Tools,Lec5 Naltural Language Processing.pdf,..\Courses\AI Tools\Lec5 Naltural Language Pro...


In [5]:
# ==========================================
# Extract Pages
# ==========================================

def extract_pages(pdf_path):

    document = pymupdf.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document, start=1):

        pages.append(
            {
                "page": page_number,
                "text": page.get_text()
            }
        )

    document.close()

    return pages

In [6]:
# ==========================================
# Build Pages Dataset
# ==========================================

pages_dataset = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pages = extract_pages(row["path"])

    for page in pages:

        pages_dataset.append(
            {
                "course": row["course"],
                "file_name": row["file_name"],
                "page": page["page"],
                "text": page["text"]
            }
        )

pages_df = pd.DataFrame(pages_dataset)

pages_df.head()

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 22.04it/s]


,course,file_name,page,text
0,AI Tools,Lec1 CV DIP Fundamentals.pdf,1,AI Tools Emerging & Technologies\nFundamental...
1,AI Tools,Lec1 CV DIP Fundamentals.pdf,2,Computer Vision\nIs a field of AI & Computer S...
2,AI Tools,Lec1 CV DIP Fundamentals.pdf,3,Computer Graphics\nPattern Recognition\nImage ...
3,AI Tools,Lec1 CV DIP Fundamentals.pdf,4,Digital Image Processing Applications\nDigital...
4,AI Tools,Lec1 CV DIP Fundamentals.pdf,5,Digital Image Processing Applications\nDigital...


In [7]:
pages_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   course     550 non-null    object
 1   file_name  550 non-null    object
 2   page       550 non-null    int64 
 3   text       550 non-null    object
dtypes: int64(1), object(3)
memory usage: 17.3+ KB


In [8]:
# ==========================================
# Build Chunks
# Chunk Size = 200 words
# Overlap = 50 words
# ==========================================

CHUNK_SIZE = 200
OVERLAP = 50

chunks = []

chunk_id = 1

for _, row in pages_df.iterrows():

    words = row["text"].split()

    start = 0

    while start < len(words):

        end = start + CHUNK_SIZE

        chunk_text = " ".join(words[start:end])

        chunks.append(
            {
                "chunk_id": chunk_id,
                "course": row["course"],
                "file_name": row["file_name"],
                "page": row["page"],
                "chunk_text": chunk_text
            }
        )

        chunk_id += 1

        start += CHUNK_SIZE - OVERLAP

chunks_df = pd.DataFrame(chunks)

chunks_df.head()

,chunk_id,course,file_name,page,chunk_text
0,1,AI Tools,Lec1 CV DIP Fundamentals.pdf,1,AI Tools Emerging & Technologies Fundamentals ...
1,2,AI Tools,Lec1 CV DIP Fundamentals.pdf,2,Computer Vision Is a field of AI & Computer Sc...
2,3,AI Tools,Lec1 CV DIP Fundamentals.pdf,3,Computer Graphics Pattern Recognition Image Pr...
3,4,AI Tools,Lec1 CV DIP Fundamentals.pdf,4,Digital Image Processing Applications Digital ...
4,5,AI Tools,Lec1 CV DIP Fundamentals.pdf,5,Digital Image Processing Applications Digital ...


In [9]:
chunks_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 556 entries, 0 to 555
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   chunk_id    556 non-null    int64 
 1   course      556 non-null    object
 2   file_name   556 non-null    object
 3   page        556 non-null    int64 
 4   chunk_text  556 non-null    object
dtypes: int64(2), object(3)
memory usage: 21.8+ KB


In [10]:
print("Number of Chunks :", len(chunks_df))

Number of Chunks : 556


In [11]:
# ==========================================
# Embedding Model
# ==========================================

from sentence_transformers import SentenceTransformer

In [12]:
# ==========================================
# Load Embedding Model
# ==========================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
sentence = "What is Machine Learning?"

embedding = embedding_model.encode(sentence)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [14]:
text1 = "Artificial Intelligence"

text2 = "AI"

emb1 = embedding_model.encode(text1)
emb2 = embedding_model.encode(text2)

In [15]:
from numpy.linalg import norm
import numpy as np

similarity = np.dot(emb1, emb2) / (norm(emb1) * norm(emb2))

print(similarity)

0.7912464


In [16]:
text1 = "Artificial Intelligence"

text2 = "Football"

In [17]:
emb1 = embedding_model.encode(text1)
emb2 = embedding_model.encode(text2)

In [18]:
similarity = np.dot(emb1, emb2) / (norm(emb1) * norm(emb2))

In [19]:
print(similarity)

0.31145296


In [20]:
# ==========================================
# Generate Embeddings
# ==========================================

embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [21]:
print(embeddings.shape)

(556, 384)


In [22]:
chunks_df["embedding"] = list(embeddings)

chunks_df.head()

,chunk_id,course,file_name,page,chunk_text,embedding
0,1,AI Tools,Lec1 CV DIP Fundamentals.pdf,1,AI Tools Emerging & Technologies Fundamentals ...,"[-0.07160597, -0.019491589, 0.011050282, -0.03..."
1,2,AI Tools,Lec1 CV DIP Fundamentals.pdf,2,Computer Vision Is a field of AI & Computer Sc...,"[-0.008366855, 0.03367389, -0.07141931, -0.110..."
2,3,AI Tools,Lec1 CV DIP Fundamentals.pdf,3,Computer Graphics Pattern Recognition Image Pr...,"[-0.05381933, 0.04813824, -0.019701764, -0.107..."
3,4,AI Tools,Lec1 CV DIP Fundamentals.pdf,4,Digital Image Processing Applications Digital ...,"[-0.029448772, 0.014464783, 0.022907672, -0.12..."
4,5,AI Tools,Lec1 CV DIP Fundamentals.pdf,5,Digital Image Processing Applications Digital ...,"[-0.029448772, 0.014464783, 0.022907672, -0.12..."


In [23]:
# ==========================================
# Import FAISS
# ==========================================

import faiss

In [24]:
embeddings = embeddings.astype("float32")

In [25]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

In [26]:
index.add(embeddings)

In [27]:
print(index.ntotal)

556


In [28]:
query = "What is Machine Learning?"

In [29]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [30]:
k = 5

distances, indices = index.search(
    query_embedding,
    k
)

In [31]:
print(indices)

[[342 343 364 317 379]]


In [32]:
results = chunks_df.iloc[indices[0]].copy()

results

,chunk_id,course,file_name,page,chunk_text,embedding
342,343,ML,Lecture3 Introduction to ML.pdf,5,What is Machine Learning? ➔The subfield of com...,"[-0.020061415, 0.018451974, 0.040135797, 0.015..."
343,344,ML,Lecture3 Introduction to ML.pdf,6,6 Computer Data Program Output Traditional Pro...,"[-0.038475573, -0.017901838, -0.008609632, -0...."
364,365,ML,Lecture3 Introduction to ML.pdf,27,27 • AI is a scientific discipline that studie...,"[-0.034298483, -0.098703876, 0.024092082, -0.0..."
317,318,ML,Lecture2.pdf,15,AI Model Learning  The process of training a ...,"[-0.021261511, -0.07354487, -0.015480035, 0.06..."
379,380,ML,Lecture3 Introduction to ML.pdf,40,Supervised learning is the types of ML in whic...,"[-0.010488525, -0.027046647, -0.039957013, 0.0..."


In [33]:
for _, row in results.iterrows():

    print("=" * 100)

    print(f"Course : {row['course']}")
    print(f"File   : {row['file_name']}")
    print(f"Page   : {row['page']}")

    print("-" * 100)

    print(row["chunk_text"])

    print("\n")

Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 5
----------------------------------------------------------------------------------------------------
What is Machine Learning? ➔The subfield of computer science that “gives computers the ability to learn without being explicitly programmed” (Arthur Samuel, 1959). ➔A computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with experience E.”(Tom Mitchell, 1997) 5


Course : ML
File   : Lecture3 Introduction to ML.pdf
Page   : 6
----------------------------------------------------------------------------------------------------
6 Computer Data Program Output Traditional Programming chart Computer Data Program Output Machine learning chart What is Machine Learning? Machinelearning It’s an application of AI Computers observe and analyze Predict based on previous patterns Model


Course : ML
File   : Lectu

In [34]:
import faiss

faiss.write_index(
    index,
    "../data/faiss_index.bin"
)

print("Index Saved Successfully")

Index Saved Successfully


In [35]:
chunks_df.to_pickle(
    "../data/chunks.pkl"
)

In [36]:
import numpy as np

np.save(
    "../data/embeddings.npy",
    embeddings
)